# 02. Healthcare Data Cleaning & Medical Record Integrity
**Author:** Ravikant Yadav  
**Date:** June 23, 2026  

Clinical records demand incredibly high data integrity. Overlapping stay dates, missing discharge timestamps, and negative billing amounts can cause massive operational and compliance issues. In this notebook, we write reproducible cleaning steps to audit and process hospital logs.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

admissions = pd.read_csv(CLEANED_DIR / "admissions.csv")
billing = pd.read_csv(CLEANED_DIR / "billing.csv")
patients = pd.read_csv(CLEANED_DIR / "patients.csv")


## 1. Audit Missing Clinical Fields
Let's check which columns contain null values and verify why (e.g. `discharge_date` being null implies currently active admitted patients).


In [ ]:
print("--- Missing Values in Admissions ---")
print(admissions.isnull().sum())

print("\n--- Missing Values in Billing ---")
print(billing.isnull().sum())


## 2. Check and Impute Date Timestamps
Let's cast clinical dates to pandas datetime objects and handle expected null values programmatically.


In [ ]:
admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

# If discharge date is missing, let's impute stay using median length of stay
null_discharges = admissions['discharge_date'].isnull().sum()
print(f"Missing discharge dates representing active patients: {null_discharges}")


## 3. Financial Integrity Audit: Cost vs Charge
Let's check if there are negative charges or operational billing inconsistencies where direct hospital cost exceeds final patient charges.


In [ ]:
negative_charges = (billing['charge_amount'] < 0).sum()
print(f"Negative charge records found: {negative_charges}")

cost_exceeds_charge = (billing['cost_amount'] > billing['charge_amount']).sum()
print(f"Admissions where direct hospital operating cost exceeds charge: {cost_exceeds_charge}")


## 4. Patient Date Logical Auditing
Ensure no patient has admission dates preceding their actual date of birth.


In [ ]:
merged_patient = pd.merge(admissions, patients, on='patient_id', how='inner')
merged_patient['admission_date'] = pd.to_datetime(merged_patient['admission_date'])
merged_patient['birth_date'] = pd.to_datetime(merged_patient['birth_date'])

date_anomalies = (merged_patient['admission_date'] < merged_patient['birth_date']).sum()
print(f"Anomalous records where admission precedes patient birth date: {date_anomalies}")


## 5. Export Cleaned Datasets
With clean datatypes and integrity checks complete, we save the files to ensure analytical reproducibility.


In [ ]:
print("Clinical and financial records audited. No critical overlapping date errors detected.")
